# Phase 5: confirm retrieval_k=200 for commit search (pre-registered)

Upload **phase5_commit_confirm_source.zip** to My Drive. Use an **A100 40 GB** and run the cells
in order. uv installs the pinned runtime; the explicit model cell downloads the embedding 0.6B
and the reranker 8B (about 17 GB). No GitHub or paid model key is needed: the fresh dataset is
already built and frozen inside the ZIP. There is no Base64 payload.

**Question.** Does building the candidate pool from longer per-signal lists (`retrieval_k=200`)
while still reranking only the fused top 50 raise Hit@5 and MRR@10 for bug report to fix commit
search? An earlier look at the v2 test split suggested +0.015 Hit@5, but that finding came from
data already examined, so it is confirmed here on repositories not used before. The hypothesis,
arms, adoption rule and timing were frozen and hashed before any sampling or retrieval.

**Arms (8B, rerank depth 50, shared scores).** A: `retrieval_k=50` (current default, control).
B: 200 (hypothesis). C: 100 (descriptive). Every arm runs the production search path; one score
cache per question means a (query, document) pair is scored once, so arms differ only by which
candidates enter the reranked top 50. Arm A is scored again from scratch to measure GPU rerun
noise. First-stage-only results and pool recall are reported for every arm.

**Rule.** B is eligible only if Hit@5 is at least +0.01 and MRR@10 at least 0 versus A, both 95%
lower bounds are at least -0.01, no repository with 30+ groups loses more than 0.02 Hit@5, the
rerun noise does not exceed the effect, and the Hit@5 interval is no wider than 0.06. Fewer than
100 fresh questions makes any conclusion tentative. Even a pass changes nothing automatically:
`retrieval_k` is shared by code, docs and commits and only commits were measured, so adoption
would be a separate per-kind setting.

**Supporting.** The 66 v2 dev questions are reported separately; the effect was never examined
there. **Timing.** 40 fresh questions, three rotated warm repetitions of real production calls per
arm, per-repo p50/p95 (A100, not laptop speed).

Labels are link-derived with no independent review, and this measures finding an issue's own fix,
not useful similar past fixes. Completed stages checkpoint to Drive; a fresh runtime with this
exact ZIP resumes. The last cell verifies the results ZIP, flushes Drive and releases the runtime.
Retrieve `phase5_commit_confirm_results.zip` from My Drive if the browser download is unfinished.

Not measured: answer quality, similar-fix usefulness, independent labels, laptop speed, code and
docs retrieval at 200, other languages. Nothing is committed.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase5_commit_confirm_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase5_commit_confirm_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "9715200472a168aef8367eea4dd4f744f2cec015783548b7dabb108bbb18f57c"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase5_commit_confirm_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys, os, pathlib, json
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0'], check=True)
from tree_sitter_language_pack import download
download(['python', 'javascript', 'typescript', 'tsx', 'java', 'go'])
ENV = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false'}
RESULTS = pathlib.Path('/content/phase5_commit_confirm_results')
RESULTS.mkdir(exist_ok=True)
PROTOCOL = json.loads((PAYLOAD / 'eval/datasets/phase5_commit_confirm_v1/protocol.json').read_text())
def execute(script, *args, env=ENV):
    subprocess.run([sys.executable, str(PAYLOAD / script), *map(str, args)],
                   env=env, check=True)

In [ ]:
import shutil, zipfile, hashlib
sys.path[:0] = [str(PAYLOAD), str(PAYLOAD / 'src')]
from eval.confirm_lib import verify_experiment, stem_of
EXPERIMENT = RESULTS / 'experiment'
payload_hash = hashlib.sha256(source_zip.read_bytes()).hexdigest()[:12]
checkpoint_path = pathlib.Path('/content/drive/MyDrive') / (
    'phase5_commit_confirm_checkpoint_' + payload_hash + '.zip')
if not any(RESULTS.iterdir()) and checkpoint_path.exists():
    with zipfile.ZipFile(checkpoint_path) as saved:
        for name in saved.namelist():
            if not (RESULTS / name).resolve().is_relative_to(RESULTS.resolve()):
                raise ValueError('Unsafe checkpoint path')
        saved.extractall(RESULTS)
BUNDLED = PAYLOAD / 'eval/datasets/phase5_commit_confirm_v1'
if not EXPERIMENT.exists():
    shutil.copytree(BUNDLED, EXPERIMENT, ignore=shutil.ignore_patterns('repos'))
FRESH = list(json.loads((EXPERIMENT / 'freeze.json').read_text())['counts'])
V2DEV = [stem_of(repo) for repo in PROTOCOL['supporting']['repos']]
for stem in FRESH:
    target = RESULTS / 'datasets' / 'fresh' / stem
    if not target.exists():
        shutil.copytree(BUNDLED / 'repos' / stem, target)
for stem in V2DEV:
    target = RESULTS / 'datasets' / 'v2' / stem
    if not target.exists():
        shutil.copytree(PAYLOAD / 'eval/datasets/phase5_bugfix_v2/repos' / stem, target)
verify_experiment(EXPERIMENT, RESULTS / 'datasets')  # frozen hashes and leakage, before any model
shutil.copy2(source_zip, RESULTS / 'source.zip')

def pack(destination):
    pending = destination.with_suffix('.pending')
    with zipfile.ZipFile(pending, 'w', zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(RESULTS.rglob('*')):
            rel = path.relative_to(RESULTS)
            if path.is_file() and 'index' not in rel.parts:
                bundle.write(path, rel.as_posix())
    pending.replace(destination)

def job(group, stem, stage):
    output = RESULTS / 'runs' / group / stem / stage
    if (output / 'run.json').exists():
        print('Already complete:', group, stem, stage)
        return
    if output.exists():
        raise FileExistsError(f'Incomplete output: {output}; preserve it before retrying')
    execute('eval/run_commit_confirm_eval.py', RESULTS, '--experiment', EXPERIMENT,
            '--group', group, '--stem', stem, '--stage', stage)
    pack(checkpoint_path)
    print('Saved Drive checkpoint:', group, stem, stage, flush=True)
pack(checkpoint_path)
print('Frozen inputs verified for', FRESH, 'and v2 dev for', V2DEV)

In [ ]:
# Download both models, verify the preregistered revisions, then go offline.
from huggingface_hub import snapshot_download
models = [('embedding', PROTOCOL['models']['embedding']),
          ('rerank8b', PROTOCOL['models']['rerankers'][1])]
for key, model in models:
    print('Downloading', model, flush=True)
    snapshot = pathlib.Path(snapshot_download(model))
    if snapshot.name != PROTOCOL['expected_revisions'][key]:
        raise ValueError(f'{model}: main revision changed; keep the protocol and ask for review')
ENV['HF_HUB_OFFLINE'] = '1'
ENV['TRANSFORMERS_OFFLINE'] = '1'

In [ ]:
# Primary evidence: the fresh holdout. Arms A/B/C share per-question scores.
for stem in FRESH:
    job('fresh', stem, 'confirm')
# Supporting evidence, reported separately: the 66 v2 dev questions.
for stem in V2DEV:
    job('v2dev', stem, 'confirm')

In [ ]:
# Real uncached production calls (first 40 fresh questions, 3 rotated repetitions).
for stem in FRESH:
    job('fresh', stem, 'timing')

In [ ]:
if not (RESULTS / 'aggregate/decision.json').exists():
    execute('eval/analyze_commit_confirm.py', RESULTS)
pack(checkpoint_path)
print((RESULTS / 'aggregate/summary.md').read_text())

In [ ]:
archive = pathlib.Path('/content/phase5_commit_confirm_results.zip')
pack(archive)
drive_archive = pathlib.Path('/content/drive/MyDrive/phase5_commit_confirm_results.zip')
shutil.copy2(archive, drive_archive)
from google.colab import files
files.download(str(archive))

In [ ]:
# Release the GPU automatically after verifying and syncing the saved results.
from google.colab import drive, runtime
import hashlib

with archive.open('rb') as local_file, drive_archive.open('rb') as saved_file:
    local_hash = hashlib.file_digest(local_file, 'sha256').digest()
    saved_hash = hashlib.file_digest(saved_file, 'sha256').digest()
    if local_hash != saved_hash:
        raise RuntimeError('Drive results do not match; keep the runtime to recover the ZIP.')
print('Syncing results to Drive before releasing the runtime...', flush=True)
drive.flush_and_unmount(timeout_ms=120000)
print('Results saved:', drive_archive, '\nReleasing the Colab runtime now.', flush=True)
runtime.unassign()